### Import Dependencies

In [1]:
import yaml
from jinja2 import Template
from langsmith import Client

### RAG Pipeline Prompt

In [2]:
def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt

In [3]:
preprocessed_context = "- a \n- b"
question = "What is a?"

In [4]:
prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

In [5]:
print(prompt)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    



### Jinja Templates

In [6]:
jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}    
"""

In [7]:
template = Template(jinja_template)

In [8]:
rendered_template = template.render(preprocessed_context=preprocessed_context, question=question)

In [9]:
print(rendered_template)


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    


In [11]:
def build_prompt_jinja(preprocessed_context, question):    
    
    jinja_template = """
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{{ preprocessed_context }}

Question:
{{ question }}    
    """

    template = Template(jinja_template)
    rendered_template = template.render(
        preprocessed_context=preprocessed_context,
        question=question
    )

    return rendered_template

In [12]:
print(build_prompt_jinja(preprocessed_context, question))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a?    
    


In [13]:
print(build_prompt_jinja("- Some item", "My silly question"))


You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- Some item

Question:
My silly question    
    


In [14]:
def prompt_template_config(yaml_path, prompt_key):

    with open(yaml_path, "r") as file:
        config = yaml.safe_load(file)

    template_content = config["prompts"][prompt_key]

    template = Template(template_content)

    return template

In [15]:
template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

In [16]:
template

<Template memory:105ed0890>

In [17]:
rendered_prompt = template.render(
    preprocessed_context=preprocessed_context,
    question=question
)

In [18]:
print(rendered_prompt)

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a? 


In [19]:
def build_prompt_jinja(preprocessed_context, question):

    template = prompt_template_config("prompts/retrieval_generation.yaml", "retrieval_generation")

    rendered_prompt = template.render(
        preprocessed_context=preprocessed_context,
        question=question
    )

    return rendered_prompt

In [20]:
print(build_prompt_jinja(preprocessed_context, question))

You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
- a 
- b

Question:
What is a? 


### Prompt Registries

In [21]:
ls_client = Client()

In [22]:
ls_template = ls_client.pull_prompt("retrieval-generation")

In [23]:
ls_template

ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, metadata={'lc_hub_owner': '-', 'lc_hub_repo': 'retrieval-generation', 'lc_hub_commit_hash': '3bf6805efd533ab6d083d2d4127b159de13c58fe7707ff360964ab1b2b9fc345'}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a shopping assistant that can answer questions about the products in stock.\nYou will be given a question and a list of context.\nInstructions:\n- Answer the question based on the provided context only.\n- Never use word context and refer to it as the available products.\n- Do not use markdown formatting.\nContext:\n{{ preprocessed_context }}\nQuestion:\n{{ question }}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='{question}'), additional_kwargs={})])

In [24]:
print(ls_template.messages[0].prompt.template)

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.
Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.
Context:
{{ preprocessed_context }}
Question:
{{ question }}


In [27]:
def prompt_template_registry(prompt_name):

    template_content = ls_client.pull_prompt(prompt_name).messages[0].prompt.template

    template = Template(template_content)

    return template

In [ ]:
print(
    prompt_template_registry("retrieval-generation").render(
        preprocessed_context=preprocessed_context,
        question=question
    )
)

You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context.
Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.
Context:
- a 
- b
Question:
What is a?


Failed to refresh cache entry retrieval-generation: Connection error caused failure to GET /commits/-/retrieval-generation/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/-/retrieval-generation/latest (Caused by NameResolutionError("HTTPSConnection(host=\'api.smith.langchain.com\', port=443): Failed to resolve \'api.smith.langchain.com\' ([Errno 8] nodename nor servname provided, or not known)"))'))
Content-Length: None
API Key: lsv2_********************************************f7
Failed to refresh cache entry retrieval-generation: Connection error caused failure to GET /commits/-/retrieval-generation/latest in LangSmith API. Please confirm your internet connection. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /commits/-/retrieval-generation/latest (Cause